<a href="https://colab.research.google.com/github/AsimaZaheer/Conference-Research-Repo/blob/main/version%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile

ZIP_PATH = "/content/drive/MyDrive/WESAD.zip"
EXTRACT_PATH = "/content/WESAD_extracted"

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"WESAD.zip nahi mili: {ZIP_PATH}\n"
        "Apni ZIP file ka correct Google Drive path set karo."
    )

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extract ho gaya!")

pkl_files = []

for root, dirs, files in os.walk(EXTRACT_PATH):
    for file in files:
        if file.endswith(".pkl") and file.startswith("S"):
            pkl_files.append(os.path.join(root, file))

print("PKL files found:", len(pkl_files))
print("Example files:", pkl_files[:5])

if not pkl_files:
    raise FileNotFoundError(
        "ZIP ke andar subject PKL files nahi milin. "
        "ZIP structure check karo."
    )

Mounted at /content/drive
Dataset extract ho gaya!
PKL files found: 15
Example files: ['/content/WESAD_extracted/WESAD/S6/S6.pkl', '/content/WESAD_extracted/WESAD/S17/S17.pkl', '/content/WESAD_extracted/WESAD/S7/S7.pkl', '/content/WESAD_extracted/WESAD/S5/S5.pkl', '/content/WESAD_extracted/WESAD/S10/S10.pkl']


In [3]:
import pickle
import numpy as np
import pandas as pd

from scipy.signal import resample_poly
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

In [4]:

from scipy.signal import butter, sosfiltfilt, find_peaks

FS_EDA = 4
FS_BVP = 64
FS_LABEL = 700

WINDOW_SECONDS = 60
STRIDE_SECONDS = 5

WINDOW_SIZE = WINDOW_SECONDS * FS_EDA
STRIDE_SIZE = STRIDE_SECONDS * FS_EDA

BVP_RATIO = FS_BVP // FS_EDA  # 16


# -----------------------------------------
# 1. EDA: statistical + tonic/phasic features
# -----------------------------------------
def extract_eda_features(signal):
    signal = np.asarray(signal, dtype=float).reshape(-1)

    if len(signal) < 20 or not np.all(np.isfinite(signal)):
        raise ValueError("Invalid EDA window")

    # Basic statistical features
    basic = [
        np.mean(signal),
        np.std(signal),
        np.min(signal),
        np.max(signal),
        np.ptp(signal)
    ]

    # Tonic component: slow changes in EDA
    tonic_sos = butter(
        2, 0.05, btype="lowpass",
        fs=FS_EDA, output="sos"
    )
    tonic = sosfiltfilt(tonic_sos, signal)

    # Phasic component: faster changes
    phasic = signal - tonic

    tonic_slope = np.polyfit(
        np.arange(len(tonic)) / FS_EDA,
        tonic, 1
    )[0]

    phasic_peaks, _ = find_peaks(
        phasic,
        distance=FS_EDA,
        prominence=max(np.std(phasic) * 0.5, 1e-8)
    )

    physiological = [
        np.mean(tonic),
        np.std(tonic),
        tonic_slope,
        np.std(phasic),
        np.mean(np.abs(phasic)),
        np.max(phasic),
        len(phasic_peaks) / WINDOW_SECONDS
    ]

    return basic + physiological


# -----------------------------------------
# 2. BVP: pulse-related features at 64 Hz
# -----------------------------------------
def extract_bvp_features(signal):
    signal = np.asarray(signal, dtype=float).reshape(-1)

    if len(signal) < 100 or not np.all(np.isfinite(signal)):
        raise ValueError("Invalid BVP window")

    # Band-pass filter to retain pulse-related frequencies
    bvp_sos = butter(
        3, [0.5, 8.0], btype="bandpass",
        fs=FS_BVP, output="sos"
    )
    filtered = sosfiltfilt(bvp_sos, signal)

    # Detect candidate pulse peaks
    peaks, _ = find_peaks(
        filtered,
        distance=int(0.3 * FS_BVP),
        prominence=max(np.std(filtered) * 0.3, 1e-8)
    )

    # Basic statistics of filtered BVP
    basic = [
        np.mean(filtered),
        np.std(filtered),
        np.min(filtered),
        np.max(filtered),
        np.ptp(filtered)
    ]

    # Pulse intervals and estimated heart rate
    if len(peaks) >= 2:
        intervals = np.diff(peaks) / FS_BVP
        heart_rates = 60.0 / intervals

        # Exclude implausible intervals for these features
        valid = (
            (intervals >= 60.0 / 200.0) &
            (intervals <= 60.0 / 40.0)
        )

        valid_intervals = intervals[valid]
        valid_hr = heart_rates[valid]

        if len(valid_hr) > 0:
            hr_mean = np.mean(valid_hr)
            hr_std = np.std(valid_hr)
            ibi_mean = np.mean(valid_intervals)
            ibi_std = np.std(valid_intervals)
        else:
            hr_mean, hr_std, ibi_mean, ibi_std = [0.0] * 4

        # Peak-to-peak pulse amplitude
        amplitudes = filtered[peaks[1:]] - filtered[peaks[:-1]]

        amp_mean = np.mean(amplitudes)
        amp_std = np.std(amplitudes)

    else:
        hr_mean, hr_std, ibi_mean, ibi_std = [0.0] * 4
        amp_mean, amp_std = 0.0, 0.0

    physiological = [
        hr_mean,
        hr_std,
        ibi_mean,
        ibi_std,
        amp_mean,
        amp_std,
        len(peaks) / WINDOW_SECONDS
    ]

    return basic + physiological


# -----------------------------------------
# 3. Load signals; preserve BVP at 64 Hz
# -----------------------------------------
def load_subject(file_path):
    with open(file_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    subject_name = os.path.basename(file_path).replace(".pkl", "")
    wrist = data["signal"]["wrist"]

    eda = np.asarray(wrist["EDA"]).reshape(-1)
    bvp_64 = np.asarray(wrist["BVP"]).reshape(-1)
    labels = np.asarray(data["label"]).reshape(-1)

    # Map each 4 Hz EDA sample to the corresponding
    # approximate label sample at 700 Hz
    label_indices = (
        np.arange(len(eda)) * (FS_LABEL / FS_EDA)
    ).astype(int)

    valid = label_indices < len(labels)
    eda = eda[:np.sum(valid)]
    label_indices = label_indices[valid]
    labels_4hz = labels[label_indices]

    # Align duration without downsampling BVP
    n = min(
        len(eda),
        len(labels_4hz),
        len(bvp_64) // BVP_RATIO
    )

    return (
        subject_name,
        eda[:n],
        bvp_64[:n * BVP_RATIO],
        labels_4hz[:n]
    )


# -----------------------------------------
# 4. Windowing and feature extraction
# -----------------------------------------
def make_windows(eda, bvp, labels, subject):
    rows = []

    n = min(len(eda), len(labels), len(bvp) // BVP_RATIO)

    for start in range(
        0, n - WINDOW_SIZE + 1, STRIDE_SIZE
    ):
        end = start + WINDOW_SIZE

        eda_window = eda[start:end]

        # Matching 60-second BVP segment at 64 Hz
        bvp_window = bvp[
            start * BVP_RATIO : end * BVP_RATIO
        ]

        label_window = labels[start:end]

        # Retain only pure Baseline or pure Stress windows
        counts = Counter(label_window.tolist())

        if counts.get(1, 0) == len(label_window):
            target = 0
        elif counts.get(2, 0) == len(label_window):
            target = 1
        else:
            continue

        try:
            eda_features = extract_eda_features(eda_window)
            bvp_features = extract_bvp_features(bvp_window)
        except ValueError:
            continue

        rows.append({
            "subject": subject,
            "label": target,
            "eda_features": eda_features,
            "bvp_features": bvp_features
        })

    return rows


In [5]:
all_rows = []

for file_path in sorted(pkl_files):
    subject = os.path.basename(file_path).replace(".pkl", "")

    # Sirf S2-S17, S12 excluded
    try:
        subject_num = int(subject[1:])
    except (ValueError, IndexError):
        continue

    if subject_num < 2 or subject_num > 17 or subject_num == 12:
        continue

    print("Processing:", subject)

    subject, eda, bvp, labels = load_subject(file_path)

    rows = make_windows(
        eda=eda,
        bvp=bvp,
        labels=labels,
        subject=subject
    )

    all_rows.extend(rows)

    print("  Valid windows:", len(rows))

if not all_rows:
    raise ValueError(
        "Koi valid window nahi bani. Dataset paths, labels aur "
        "signal lengths check karo."
    )

print("\nTotal valid windows:", len(all_rows))
print("Subjects:", sorted(set(r["subject"] for r in all_rows)))

print("\nClass distribution:")
print(pd.Series([r["label"] for r in all_rows])
      .map({0: "Baseline", 1: "Stress"})
      .value_counts())

Processing: S10
  Valid windows: 357
Processing: S11
  Valid windows: 348
Processing: S13
  Valid windows: 345
Processing: S14
  Valid windows: 347
Processing: S15
  Valid windows: 349
Processing: S16
  Valid windows: 347
Processing: S17
  Valid windows: 357
Processing: S2
  Valid windows: 328
Processing: S3
  Valid windows: 332
Processing: S4
  Valid windows: 334
Processing: S5
  Valid windows: 345
Processing: S6
  Valid windows: 342
Processing: S7
  Valid windows: 341
Processing: S8
  Valid windows: 344
Processing: S9
  Valid windows: 341

Total valid windows: 5157
Subjects: ['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']

Class distribution:
Baseline    3342
Stress      1815
Name: count, dtype: int64


In [6]:

X_eda = np.array([
    r["eda_features"] for r in all_rows
], dtype=float)

X_fusion = np.array([
    r["eda_features"] + r["bvp_features"]
    for r in all_rows
], dtype=float)

y = np.array([r["label"] for r in all_rows])
groups = np.array([r["subject"] for r in all_rows])

print("EDA feature matrix:", X_eda.shape)
print("EDA+BVP feature matrix:", X_fusion.shape)
print("Labels:", y.shape)

logo = LeaveOneGroupOut()

experiment_results = []
subject_results = []

experiments = {
    "EDA + Random Forest": X_eda,
    "EDA + BVP + Random Forest": X_fusion
}

for experiment_name, X in experiments.items():
    print("\n" + "=" * 60)
    print("Experiment:", experiment_name)
    print("=" * 60)

    fold_accuracies = []
    fold_balanced_accuracies = []
    fold_f1_scores = []

    for train_idx, test_idx in logo.split(X, y, groups):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        test_subject = groups[test_idx][0]

        # Training data mein dono classes honi chahiye
        if len(np.unique(y_train)) < 2:
            print("Skipping fold:", test_subject,
                  "- training mein ek hi class hai.")
            continue

        model = RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        acc = accuracy_score(y_test, predictions)
        bal_acc = balanced_accuracy_score(y_test, predictions)
        macro_f1 = f1_score(
            y_test,
            predictions,
            labels=[0, 1],
            average="macro",
            zero_division=0
        )

        fold_accuracies.append(acc)
        fold_balanced_accuracies.append(bal_acc)
        fold_f1_scores.append(macro_f1)

        subject_results.append({
            "Experiment": experiment_name,
            "Test Subject": test_subject,
            "Accuracy": acc,
            "Balanced Accuracy": bal_acc,
            "Macro F1": macro_f1
        })

        print(
            f"{test_subject}: "
            f"Accuracy={acc:.4f}, "
            f"Balanced Accuracy={bal_acc:.4f}, "
            f"Macro F1={macro_f1:.4f}"
        )

    experiment_results.append({
        "Experiment": experiment_name,
        "Accuracy Mean": np.mean(fold_accuracies),
        "Accuracy SD": np.std(fold_accuracies, ddof=1),
        "Balanced Accuracy Mean": np.mean(fold_balanced_accuracies),
        "Balanced Accuracy SD": np.std(fold_balanced_accuracies, ddof=1),
        "Macro F1 Mean": np.mean(fold_f1_scores),
        "Macro F1 SD": np.std(fold_f1_scores, ddof=1),
        "Valid Folds": len(fold_accuracies)
    })

summary_df = pd.DataFrame(experiment_results)
subject_df = pd.DataFrame(subject_results)

print("\n\nOVERALL RESULTS")
display(summary_df.round(4))

print("\nSUBJECT-WISE RESULTS")
display(subject_df.round(4))

EDA feature matrix: (5157, 12)
EDA+BVP feature matrix: (5157, 24)
Labels: (5157,)

Experiment: EDA + Random Forest
S10: Accuracy=0.8431, Balanced Accuracy=0.7910, Macro F1=0.8121
S11: Accuracy=0.6092, Balanced Accuracy=0.6910, Macro F1=0.6063
S13: Accuracy=0.7391, Balanced Accuracy=0.7991, Macro F1=0.7388
S14: Accuracy=0.6455, Balanced Accuracy=0.5000, Macro F1=0.3923
S15: Accuracy=0.8510, Balanced Accuracy=0.8161, Macro F1=0.8299
S16: Accuracy=0.9625, Balanced Accuracy=0.9472, Macro F1=0.9580
S17: Accuracy=0.7255, Balanced Accuracy=0.6973, Macro F1=0.7006
S2: Accuracy=0.8902, Balanced Accuracy=0.8400, Macro F1=0.8658
S3: Accuracy=0.7048, Balanced Accuracy=0.7013, Macro F1=0.6894
S4: Accuracy=0.9102, Balanced Accuracy=0.8696, Macro F1=0.8929
S5: Accuracy=0.7159, Balanced Accuracy=0.7248, Macro F1=0.7034
S6: Accuracy=0.8187, Balanced Accuracy=0.7794, Macro F1=0.7902
S7: Accuracy=0.8944, Balanced Accuracy=0.8490, Macro F1=0.8727
S8: Accuracy=0.8605, Balanced Accuracy=0.8217, Macro F1=0.8

,Experiment,Accuracy Mean,Accuracy SD,Balanced Accuracy Mean,Balanced Accuracy SD,Macro F1 Mean,Macro F1 SD,Valid Folds
0,EDA + Random Forest,0.8087,0.1114,0.7845,0.1120,0.7762,0.1463,15
1,EDA + BVP + Random Forest,0.8026,0.1559,0.7852,0.1554,0.7618,0.1967,15



SUBJECT-WISE RESULTS


,Experiment,Test Subject,Accuracy,Balanced Accuracy,Macro F1
0,EDA + Random Forest,S10,0.8431,0.7910,0.8121
1,EDA + Random Forest,S11,0.6092,0.6910,0.6063
2,EDA + Random Forest,S13,0.7391,0.7991,0.7388
3,EDA + Random Forest,S14,0.6455,0.5000,0.3923
4,EDA + Random Forest,S15,0.8510,0.8161,0.8299
5,EDA + Random Forest,S16,0.9625,0.9472,0.9580
6,EDA + Random Forest,S17,0.7255,0.6973,0.7006
7,EDA + Random Forest,S2,0.8902,0.8400,0.8658
8,EDA + Random Forest,S3,0.7048,0.7013,0.6894
9,EDA + Random Forest,S4,0.9102,0.8696,0.8929


In [ ]:
summary_df.to_csv(
    "/content/WESAD_overall_results.csv",
    index=False
)

subject_df.to_csv(
    "/content/WESAD_subject_results.csv",
    index=False
)

print("Results saved successfully.")

from google.colab import files

files.download("/content/WESAD_overall_results.csv")
files.download("/content/WESAD_subject_results.csv")

Results saved successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>